In [0]:
from pyspark.sql import functions as F

staging_orders = spark.table("brazilian_ecommerce.prod.staging_orders")

gold_delivery_performance = (
    staging_orders
    .withColumn("year_month", F.date_format("order_purchase_timestamp", "yyyy-MM"))
    .groupBy("year_month")
    .agg(
        F.count(F.when(F.col("delivery_status").isNotNull(), 1)).alias("completed_orders"),
        F.count(F.when(F.col("delivery_status") == "on_time", 1)).alias("on_time_orders"),
        F.count(F.when(F.col("delivery_status") == "delayed", 1)).alias("delayed_orders"),
        F.avg("hours_purchase_to_approved").alias("avg_hours_purchase_to_approved"),
        F.avg("hours_approved_to_carrier").alias("avg_hours_approved_to_carrier"),
        F.avg("hours_carrier_to_customer").alias("avg_hours_carrier_to_customer"),
        F.avg("hours_estimated_to_delivered").alias("avg_hours_estimated_to_delivered"),
    )
    .withColumn(
        "pct_on_time",
        F.when(
            F.col("completed_orders") > 0,
            (F.col("on_time_orders") / F.col("completed_orders")).cast("decimal(5,4)")
        )
    )
    .withColumn(
        "pct_delayed",
        F.when(
            F.col("completed_orders") > 0,
            (F.col("delayed_orders") / F.col("completed_orders")).cast("decimal(5,4)")
        )
    )
    .drop("on_time_orders", "delayed_orders")
    .withColumn("_loaded_at", F.current_timestamp())
    .orderBy("year_month")
)

full_table_name = "brazilian_ecommerce.prod.gold_delivery_performance"

(
    gold_delivery_performance.write
    .mode("overwrite")
    .option("comment", "gold table: monthly delivery speed and reliability metrics")
    .option("overwriteSchema", "true")
    .saveAsTable(full_table_name)
)

print(f"Loaded {gold_delivery_performance.count()} rows into {full_table_name}")